# Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of multiple-choice questions one at a time. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:
$$Y_i = \begin{cases} 1, & \text{if the user answers item } i \text{ correctly,} \\ 0, & \text{if the user answers item } i \text{ incorrectly.} \end{cases}$$

The response probability for item $i$ under the two-parameter logistic (2PL) item response model is:
$$P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

where $a_i > 0$ is the discrimination parameter and $b_i$ is the difficulty parameter.

## Task 1: Visualizing 2PL IRT Curves

We plot $P(Y_i = 1 \mid \Theta = \theta)$ vs. $\theta$ across various combinations of discrimination ($a_i$) and difficulty ($b_i$) parameters.

* **Horizontal Shift ($b_i$):** Changing $b_i$ shifts the item characteristic curve horizontally. A larger $b_i$ moves the curve to the right, requiring higher ability $\theta$ for a $50\%$ chance of a correct response.
* **Slope/Steepness ($a_i$):** Higher values of $a_i$ make the slope steeper at $\theta = b_i$, increasing the item's ability to discriminate between users around that ability level.

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]
    p_vals = p_i(theta_vals, a, b)

    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

fig.update_layout(
    title={'text': "Two-Parameter Logistic (2PL) Item Response Curves", 'y':0.9, 'x':0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05, bgcolor="rgba(255,255,255,0.8)")
)

fig.show()

## Tasks 2–5: Mathematical Formulation & Concepts

### Task 2: Likelihood Contribution
* **Single Response Likelihood:**
  $$L(y_k \mid \theta) = p_k(\theta)^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$
* **Joint Likelihood:**
  $$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} p_i(\theta)^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

### Task 3: Running Posterior Update
The running recursive update equation is:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Normalized explicitly:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{p_k(\theta)^{y_k}[1 - p_k(\theta)]^{1-y_k} f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{\mathbb{R}} p_k(s)^{y_k}[1 - p_k(s)]^{1-y_k} f_{\Theta \mid \mathbf{Y}^{(k-1)}}(s \mid \mathbf{y}^{(k-1)}) \, ds}$$

### Task 4: Dynamic Shifting
When $y_k = 1$ on a difficult item (large $b_k$), the likelihood weight $p_k(\theta)$ is close to zero for low values of $\theta$ and grows rapidly around $\theta \approx b_k$. Multiplying the prior density by this steep upward curve significantly suppresses lower ability values, shifting the peak (mode) of the posterior density to the right.

### Task 5: Certainty and Sharpness ($a_k$)
The discrimination parameter $a_k$ determines how steeply the likelihood transitions from 0 to 1.
* **Large $a_k$:** Creates a sharp transition curve, injecting high information and significantly narrowing (reducing variance of) the posterior density.
* **Small $a_k$:** Creates a flat transition curve, contributing little information and leaving posterior variance mostly unchanged.

## Tasks 6 & 7: Grid Approximation & Convergence Simulation

We discretize $\theta$ over a fine grid and update the running posterior sequentially using numerical integration (`np.trapezoid`). Point estimates are maintained at each step:
* **Posterior Mean ($\hat{\theta}_{\text{Bayes}}$):** Computed via numerical integration $\int \theta \, f(\theta \mid \mathbf{y}) \, d\theta$.
* **MAP Estimate ($\hat{\theta}_{\text{MAP}}$):** Extracted via $\arg\max_\theta f(\theta \mid \mathbf{y})$.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

# Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

fig2 = go.Figure()

fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})", annotation_position="bottom right"
)

fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)', line=dict(color='blue', width=2.5), marker=dict(size=6)
))

fig2.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)', line=dict(color='green', width=2), marker=dict(size=6, symbol='square')
))

fig2.update_layout(
    title={'text': "Convergence of Latent Ability Estimators (θ) Over Time", 'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

fig2.show()

# Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

We track the click-through rate $\theta \in [0, 1]$ of an ad using continuous Bernoulli trials $Y_k \in \{0, 1\}$.

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0, 1, 500)

beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State: Beta(1,1)", "color": "blue", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State: Beta(2,8)", "color": "red", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State: Beta(8,2)", "color": "green", "dash": "solid"}
]

fig = go.Figure()

for config in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid, config["alpha"], config["beta"])
    fig.add_trace(go.Scatter(
        x=theta_grid, y=pdf_vals, mode='lines', name=config["name"],
        line=dict(color=config["color"], dash=config["dash"], width=2.5)
    ))

fig.update_layout(
    title={'text': "Structural Variations of the Beta(α, β) PDF", 'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Parameter Value (θ)", yaxis_title="Probability Density f(θ)",
    template="plotly_white", hovermode="x unified"
)

fig.show()

## Tasks 2–5: Beta-Binomial Derivations

### Task 2: Likelihood Functions
* **Single Observation:** $L(y_k \mid \theta) = \theta^{y_k}(1 - \theta)^{1 - y_k}$
* **Joint History Likelihood:**
  $$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i}(1-\theta)^{1-y_i} = \theta^{C_k}(1-\theta)^{k - C_k}$$
  where $C_k = \sum_{i=1}^k y_i$ is the running count of clicks.

### Task 3: Conjugate Beta Parameters & Posterior Mean
Combining Beta prior $\text{Beta}(\alpha_{k-1}, \beta_{k-1})$ with Bernoulli likelihood:
$$f(\theta \mid \mathbf{y}^{(k)}) \propto \left[\theta^{y_k}(1-\theta)^{1-y_k}\right] \left[\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}\right] = \theta^{(\alpha_{k-1}+y_k)-1}(1-\theta)^{(\beta_{k-1}+1-y_k)-1}$$

Closed-form parameter updates:
$$\alpha_k = \alpha_{k-1} + y_k, \quad \beta_k = \beta_{k-1} + (1 - y_k)$$

Analytical Posterior Mean:
$$\mathbb{E}[\Theta \mid \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + C_k}{\alpha_0 + \beta_0 + k}$$

### Task 5: Closed-Form Point Estimators
* **Posterior Mean:** $\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$
* **MAP Estimate:** $\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (for $\alpha_k, \beta_k > 1$)

In [4]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.35
n_impressions = 100
steps = list(range(n_impressions + 1))

alpha_param = 1
beta_param = 1

running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha_param += y_k
    beta_param += (1 - y_k)

    theta_bayes_k = alpha_param / (alpha_param + beta_param)
    theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2) if (alpha_param > 1 and beta_param > 1) else (0.0 if alpha_param <= beta_param else 1.0)

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

fig2 = go.Figure()

fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right"
)

fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines', name='Exact Posterior Mean', line=dict(color='blue', width=2.5)))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode='lines', name='Exact MAP Estimate', line=dict(color='green', width=1.5, dash='dot')))

fig2.update_layout(
    title={'text': "Analytical Beta-Binomial Conjugate Update Timeline", 'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Number of User Impressions (k)", yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white"
)

fig2.show()

# Structural Health Monitoring via Bounded Grid Updates

Estimating structural stiffness degradation $\theta \in (0, 1]$ from continuous, log-normally corrupted stiffness sensor readings:
$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_sensor_readings = 15

theta_grid = np.linspace(0.01, 1.0, 500)
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
current_posterior /= np.trapezoid(current_posterior, theta_grid)

milestones = [0, 1, 2, 5, 10, 15]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior, mode='lines',
    name='Prior State: Assumed Healthy', line=dict(dash='dash', width=2.5, color='gray')
))

for k in range(1, n_sensor_readings + 1):
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    current_posterior *= likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior, mode='lines',
            name=f"Step {k}: (Observed K={y_k:.2f})", line=dict(width=2)
        ))

fig.add_vline(x=theta_true, line_dash="dot", line_color="red", line_width=2.5, annotation_text=f"True State ({theta_true})")
fig.update_layout(
    title={'text': "Structural Health Monitoring: Grid Parameter Update", 'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},
    xaxis_title="Remaining Stiffness Factor (θ)", yaxis_title="Probability Density",
    template="plotly_white"
)

fig.show()

# Gaussian Mixture Clustering as Conditional Updating

## Analytical Proofs (Parts 1–9)

### Part 1: Marginal Density
$$p(\mathbf{x}_i) = \sum_{k=1}^{K} P(C_i = k) p(\mathbf{x}_i \mid C_i = k) = \sum_{k=1}^{K} \phi_k \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

### Part 2: Posterior Responsibilities
Using Bayes' Rule:
$$\gamma_{ik} = P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i) = \frac{\phi_k \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K} \phi_j \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

### Part 3: Soft Assignment Expectation
For the binary indicator vector $\mathbf{Z}_i$, $\mathbb{E}[Z_{ik} \mid \mathbf{X}_i = \mathbf{x}_i] = P(C_i = k \mid \mathbf{X}_i = \mathbf{x}_i) = \gamma_{ik}$.
$$\mathbb{E}[\mathbf{Z}_i \mid \mathbf{X}_i = \mathbf{x}_i] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$

### Part 7 & 8: EM Update Equations
* **Effective Count:** $N_k = \sum_{i=1}^n \gamma_{ik}$
* **Weight:** $\phi_k^{\text{new}} = \frac{N_k}{n}$
* **Mean:** $\boldsymbol{\mu}_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} \mathbf{x}_i$
* **Covariance:** $\boldsymbol{\Sigma}_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})(\mathbf{x}_i - \boldsymbol{\mu}_k^{\text{new}})^T$

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
        )

    def prepare_data(self, df, feature_cols, test_size=0.2):
        X = df[feature_cols].dropna().values
        X_scaled = self.scaler.fit_transform(X)
        return train_test_split(X_scaled, test_size=test_size, random_state=self.random_state)

    def fit(self, X_train):
        self.model.fit(X_train)
        print("GMM Training complete.")
        print(f"Converged: {self.model.converged_}")
        print(f"Iterations taken: {self.model.n_iter_}")

    def evaluate(self, X_test):
        avg_log_likelihood = self.model.score(X_test)
        print(f"Average Log-Likelihood (Test Set): {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def _generate_contour_base(self, X_data):
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        responsibilities = self.model.predict_proba(grid_points)
        max_prob = responsibilities.max(axis=1).reshape(xx.shape)
        grid_orig = self.scaler.inverse_transform(grid_points)
        return grid_orig[:, 0].reshape(xx.shape), grid_orig[:, 1].reshape(yy.shape), max_prob

    def plot_training_assignments(self, X_train, feature_names):
        xx_orig, yy_orig, max_prob = self._generate_contour_base(X_train)
        hard_labels = self.model.predict(X_train)
        X_train_orig = self.scaler.inverse_transform(X_train)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob,
            colorscale="Cividis", contours_coloring="heatmap", opacity=0.6, name="Confidence"
        ))

        for k in range(self.n_components):
            mask = hard_labels == k
            fig.add_trace(go.Scatter(
                x=X_train_orig[mask, 0], y=X_train_orig[mask, 1],
                mode="markers", name=f"Cluster {k+1}", marker=dict(size=6)
            ))

        fig.update_layout(
            title="GMM Soft-Assignment Confidence Boundaries on Training Data",
            xaxis_title=feature_names[0], yaxis_title=feature_names[1], template="plotly_white"
        )
        fig.show()

# Synthetic test harness execution
if __name__ == "__main__":
    np.random.seed(42)
    synthetic_data = np.vstack([
        np.random.multivariate_normal([1000, 2000], [[500000, 200000], [200000, 500000]], 300),
        np.random.multivariate_normal([5000, 8000], [[1000000, -300000], [-300000, 1000000]], 300),
        np.random.multivariate_normal([50, 500], [[10000, 2000], [2000, 20000]], 300)
    ])
    df_demo = pd.DataFrame(synthetic_data, columns=["PURCHASES", "CREDIT_LIMIT"])

    segmenter = GMMFinancialSegmenter(n_components=3)
    X_train, X_test = segmenter.prepare_data(df_demo, ["PURCHASES", "CREDIT_LIMIT"])
    segmenter.fit(X_train)
    segmenter.evaluate(X_test)
    segmenter.plot_training_assignments(X_train, ["PURCHASES", "CREDIT_LIMIT"])

GMM Training complete.
Converged: True
Iterations taken: 6
Average Log-Likelihood (Test Set): -0.4851
